# 01 — Data Pipeline & Quality Check

This notebook validates the raw FINRA and price data downloaded by .  It establishes the universe, checks for gaps and outliers, and produces summary statistics that inform downstream signal research.

> **Note on borrow rate data**: Actual transaction borrow rates (from EquiLend / S3 Partners) are proprietary. This notebook uses the utilisation-based proxy described in .

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src")))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import date

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "#F8F9FA",
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

from securities_lending.ingestion import FINRARegSHOIngester, FINRAShortInterestIngester, PriceIngester
from securities_lending.utils.config import load_universe

uni = load_universe()
TICKERS = uni["universe"]["tickers"]
START = uni["data"]["start_date"]
print(f"Universe: {len(TICKERS)} tickers, data from {START}")

## 1. FINRA Reg SHO — Daily Short Sale Volume

The SVR (short volume ratio) is the cleanest high-frequency signal in this dataset: it reflects the fraction of consolidated NMS volume executed as a short sale on a given day, excluding exempt (market-maker) shorts.

In [ ]:
regsho = FINRARegSHOIngester(cache_dir="../data/raw/finra_regsho", tickers=TICKERS)
svr_panel = regsho.load_panel(start=START)
print(f"SVR panel: {svr_panel.shape[0]} dates × {svr_panel.shape[1]} tickers")
print(f"
Date range: {svr_panel.index.min()} — {svr_panel.index.max()}")
print(f"
Mean SVR across universe: {svr_panel.mean().mean():.4f}")
print(f"SVR coverage (non-NaN): {svr_panel.notna().mean().mean():.1%}")

In [ ]:
# Distribution of SVR by ticker (latest 252 days)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
svr_latest = svr_panel.tail(252)
svr_means = svr_latest.mean().dropna()

axes[0].hist(svr_means, bins=40, color="#2B6CB0", alpha=0.8, edgecolor="white")
axes[0].axvline(svr_means.mean(), color="#C53030", linewidth=2,
                label=f"Mean={svr_means.mean():.3f}")
axes[0].set_title("Distribution of Mean SVR per Ticker (trailing 252d)")
axes[0].set_xlabel("Mean Short Volume Ratio")
axes[0].set_ylabel("Count")
axes[0].legend()

# Time series of universe-average SVR
svr_univ = svr_panel.mean(axis=1)
svr_univ.plot(ax=axes[1], color="#718096", alpha=0.6, linewidth=0.8, label="Daily")
svr_univ.rolling(21).mean().plot(ax=axes[1], color="#2B6CB0", linewidth=2, label="21d MA")
axes[1].set_title("Universe-Average SVR Over Time")
axes[1].set_ylabel("Mean SVR")
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

## 2. FINRA Short Interest — Biweekly Positions

In [ ]:
si_ingester = FINRAShortInterestIngester(cache_dir="../data/raw/finra_short_interest", tickers=TICKERS)
si_daily = si_ingester.load_daily_interpolated(start=START)
if not si_daily.empty:
    print(f"SI daily (interpolated): {len(si_daily)} rows, {si_daily['symbol'].nunique()} tickers")
    print(si_daily.head(10))
else:
    print("No SI data loaded — run fetch_data.py first")

## 3. Price Data Quality

In [ ]:
price_ingester = PriceIngester(cache_dir="../data/raw/prices")
prices = price_ingester.load(tickers=TICKERS, start=START)
print(f"Price data: {len(prices)} rows, {prices['symbol'].nunique()} tickers")
print(f"Date range: {prices['date'].min()} — {prices['date'].max()}")

# Coverage heatmap
coverage = prices.pivot_table(index="date", columns="symbol", values="close", aggfunc="count")
coverage_pct = coverage.notna().mean(axis=1)
fig, ax = plt.subplots(figsize=(12, 3))
coverage_pct.plot(ax=ax, color="#276749")
ax.set_title("Fraction of Universe with Price Data by Date")
ax.set_ylabel("Coverage")
ax.set_ylim(0, 1.1)
plt.show()

## 4. Known Event Validation — GME Short Squeeze (Jan 2021)

A key sanity check: the pipeline should capture the GME event clearly in SVR and DTC metrics.

In [ ]:
# Spot check GME around Jan 2021
gme_prices = prices[prices["symbol"] == "GME"].set_index("date")
gme_svr = svr_panel.get("GME", pd.Series(dtype=float))

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
gme_prices["close"].plot(ax=axes[0], color="#2B6CB0", linewidth=1.5)
axes[0].set_title("GME Price")
axes[0].set_ylabel("Price (USD)")

if not gme_svr.empty:
    gme_svr.plot(ax=axes[1], color="#C53030", alpha=0.7, linewidth=0.8)
    gme_svr.rolling(5).mean().plot(ax=axes[1], color="#C53030", linewidth=2, label="5d MA")
    axes[1].set_title("GME Short Volume Ratio (SVR)")
    axes[1].set_ylabel("SVR")
    axes[1].axvspan(pd.Timestamp("2021-01-13"), pd.Timestamp("2021-02-05"),
                    alpha=0.15, color="orange", label="Squeeze period")
    axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()
print("
Expected: SVR elevated in Dec 2020 / Jan 2021 before and during the squeeze")